In [8]:
import re
from urllib.request import urlopen
from html.parser import HTMLParser
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

URL = "https://www.gutenberg.org/browse/scores/top#books-last1"
OUTPUT_DIR = Path("../data/books")
OUTPUT_DIR.mkdir(exist_ok=True)

# 🧠 Parser HTML simple
class BookParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.in_section = False
        self.capture = False
        self.ids = []

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)

        if tag == "h2":
            self.capture = True

        if tag == "a" and self.in_section:
            href = attrs.get("href", "")
            match = re.search(r"/ebooks/(\d+)", href)
            if match:
                self.ids.append(match.group(1))

    def handle_data(self, data):
        if self.capture and "Top 100 EBooks yesterday" in data:
            self.in_section = True
            self.capture = False

    def handle_endtag(self, tag):
        if tag == "ol" and self.in_section:
            self.in_section = False


# 🌐 Obtener HTML
print("🌐 Cargando página...")
html = urlopen(URL).read().decode("utf-8")

parser = BookParser()
parser.feed(html)

book_ids = list(set(parser.ids))
print(f"📚 Libros encontrados: {len(book_ids)}")

# 🔗 URLs posibles
def build_urls(book_id):
    return [
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt",
    ]

# 🧼 limpiar texto
def clean_text(text):
    start = re.search(r"\*\*\* START OF.*?\*\*\*", text)
    end = re.search(r"\*\*\* END OF.*?\*\*\*", text)
    if start and end:
        return text[start.end():end.start()].strip()
    return text

# ⬇️ descargar libro
def download(book_id):
    for url in build_urls(book_id):
        try:
            data = urlopen(url, timeout=10).read().decode("utf-8", errors="ignore")
            if len(data) > 1000:
                clean = clean_text(data)
                (OUTPUT_DIR / f"{book_id}.txt").write_text(clean, encoding="utf-8")
                return f"✅ {book_id}"
        except:
            continue
    return f"❌ {book_id}"

# ⚡ paralelo
print("⬇️ Descargando...")
results = []

with ThreadPoolExecutor(max_workers=10) as ex:
    futures = [ex.submit(download, bid) for bid in book_ids]

    for f in as_completed(futures):
        r = f.result()
        print(r)
        results.append(r)

# 📊 resumen
ok = sum(1 for r in results if "✅" in r)
fail = sum(1 for r in results if "❌" in r)

print("\n📊 RESULTADO FINAL")
print(f"✅ {ok} descargados")
print(f"❌ {fail} fallidos")

🌐 Cargando página...
📚 Libros encontrados: 100
⬇️ Descargando...
✅ 11
✅ 1513
✅ 27673
✅ 468
✅ 22541
✅ 21839
✅ 2148
✅ 15399
✅ 75201
✅ 831
✅ 1661
✅ 244
✅ 3268
✅ 9701
✅ 161
✅ 1998
✅ 74
✅ 2554
✅ 28054
✅ 768
✅ 1259
✅ 78509
✅ 1727
✅ 829
✅ 23
✅ 5197
✅ 1342
✅ 1260
✅ 36034
✅ 34413
✅ 6761
✅ 58031
✅ 8492
❌ 28942
✅ 174
✅ 4300
✅ 51252
✅ 100
✅ 64317
✅ 2160
✅ 59828
✅ 209
✅ 4363
✅ 78508
✅ 84
✅ 2465
✅ 6593
✅ 2868
✅ 1080
✅ 67979
✅ 1232
✅ 2680
✅ 2591
✅ 3011
✅ 8800
✅ 29090
✅ 55
✅ 62215
✅ 8438
✅ 2542
✅ 65238
✅ 3207
✅ 43
✅ 394
✅ 76
✅ 72
✅ 1695
✅ 844
✅ 145
✅ 45
✅ 205
✅ 71046
✅ 78514
✅ 2701
✅ 120
✅ 42671
✅ 3296
✅ 2600
✅ 1400
✅ 37106
✅ 98
✅ 46976
✅ 2852
✅ 2641
✅ 25344
✅ 50559
✅ 6400
✅ 601
✅ 4085
✅ 52106
✅ 1952
✅ 16
✅ 1184
✅ 20203
✅ 5200
✅ 16389
✅ 16328
✅ 6133
✅ 345
✅ 45304

📊 RESULTADO FINAL
✅ 99 descargados
❌ 1 fallidos
